
# CRE Valuation Gap – Policy Transmission Analysis

This notebook implements a **clean, coherent pipeline**:

1. Data Preparation  
2. Feature Engineering  
3. Exploratory Analysis  
4. OLS (Mean Model)  
5. Quantile Regression (Distributional Model)  
6. Key Insights  

Focus: Understanding how **ΔRepo affects valuation gap dynamics across regimes and distribution tails**.


In [ ]:

import pandas as pd
import numpy as np
import statsmodels.api as sm
import statsmodels.formula.api as smf
import matplotlib.pyplot as plt
import seaborn as sns


## Load Data

In [ ]:

# Replace with your actual file
df = pd.read_csv("your_data.csv")

df.head()


## Feature Engineering

In [ ]:

# Delta repo (policy shock)
df["delta_repo"] = df["repo_rate"].diff()

# Valuation gap (if not already present)
# df["valuation_gap"] = df["hpi_growth"] - df["rent_growth"]

# Delta valuation gap
df["delta_valuation_gap"] = df["valuation_gap"].diff()

# Regime dummies (example – adjust thresholds)
df["low_regime"] = (df["regime"] == "low").astype(int)
df["high_regime"] = (df["regime"] == "high").astype(int)

df = df.dropna()

df.head()


## Exploratory Analysis

In [ ]:

corr = df.corr(numeric_only=True)
print(corr["delta_valuation_gap"].sort_values(ascending=False))


## OLS Model (Mean Effect)

In [ ]:

model = smf.ols(
    "delta_valuation_gap ~ delta_repo + delta_repo:high_regime + delta_repo:low_regime",
    data=df
).fit()

print(model.summary())


## Quantile Regression

In [ ]:

quantiles = [0.1, 0.5, 0.9]

for q in quantiles:
    print(f"\n===== Quantile {q} =====")
    mod = smf.quantreg(
        "delta_valuation_gap ~ delta_repo + delta_repo:high_regime + delta_repo:low_regime",
        df
    )
    res = mod.fit(q=q)
    print(res.summary())


## Visualization

In [ ]:

sns.lmplot(
    data=df,
    x="delta_repo",
    y="valuation_gap",
    hue="regime",
    height=5,
    aspect=1.2
)
plt.title("ΔRepo vs Valuation Gap by Regime")
plt.show()



## Key Findings

- OLS → No mean effect  
- Median (0.5) → No effect  
- Tails (0.1, 0.9) → Significant effects in **low regime**

### Interpretation

- Policy is **not a mean driver**
- Policy acts as a **tail regulator**
- Strong markets absorb policy
- Weak markets react strongly

---

## Final Conclusion

**Monetary policy affects valuation through distributional channels, not average outcomes.**
